# Análisis de dinámica molecular — GO19–glifosato–FAD (revisión Frontiers)
### WT · R53A · R272L · R53A/R272L — 3 réplicas independientes × 100 ns por sistema

**Cómo usarlo**
1. Corre la celda **⚙️ Configuración** (ajusta formato, dpi, tipografía, paleta, etiquetas de ejes).
2. Corre todo (*Entorno de ejecución → Ejecutar todas*). La primera vez pedirá acceso a Google Drive.
3. Todo se guarda en **`MyDrive/go19_md_results_redo/`**: `figures/`, `tables/`, `captions/`.
4. Para cambiar el estilo: modifica la configuración y vuelve a correr solo las celdas de las figuras.

Datos: `GO19_colab.zip` **versión 0–100 ns** (CSV de MDAnalysis, ProLIF 1001 frames/réplica, MM-GBSA 251 frames/réplica, NAC y comparación de ventanas, generados en la PC de análisis). Colócalo en la carpeta de Drive
`go19_md_results_redo/` o súbelo cuando se pida.

Convenciones: todas las medias sobre la trayectoria completa de producción (0–100 ns); media de 3 réplicas ± SD muestral (n − 1); en series de tiempo, línea gruesa = media,
banda = intervalo mín–máx entre réplicas, líneas finas = réplicas.

## ⚙️ Configuración

In [ ]:
#@title ⚙️ Configuración — edita y vuelve a correr
#@markdown **Salida**
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "go19_md_results_redo"  #@param {type:"string"}
FORMAT = "png + tiff"  #@param ["png", "tiff", "svg", "pdf", "png + tiff", "png + svg", "png + pdf", "svg + pdf", "all"]
DPI_LIST = "300, 600"  #@param {type:"string"}
TIFF_COMPRESSION = "tiff_lzw"  #@param ["tiff_lzw", "none"]
#@markdown **Tipografía**
FONT_FAMILY = "Arial"  #@param ["Arial", "Helvetica", "Times New Roman", "DejaVu Sans", "DejaVu Serif"]
AXIS_LABEL_SIZE = 11  #@param {type:"slider", min:6, max:20, step:0.5}
AXIS_LABEL_WEIGHT = "normal"  #@param ["normal", "bold"]
AXIS_LABEL_STYLE = "normal"  #@param ["normal", "italic"]
TICK_LABEL_SIZE = 9  #@param {type:"slider", min:5, max:18, step:0.5}
TICK_LABEL_WEIGHT = "normal"  #@param ["normal", "bold"]
LEGEND_SIZE = 8  #@param {type:"slider", min:5, max:16, step:0.5}
LEGEND_WEIGHT = "normal"  #@param ["normal", "bold"]
LEGEND_STYLE = "normal"  #@param ["normal", "italic"]
PANEL_LETTERS = True  #@param {type:"boolean"}
PANEL_LETTER_SIZE = 12  #@param {type:"slider", min:6, max:24, step:0.5}
PANEL_LETTER_WEIGHT = "bold"  #@param ["normal", "bold"]
PANEL_LETTER_STYLE = "normal"  #@param ["normal", "italic"]
#@markdown **Trazos y tamaño**
LINE_WIDTH = 1.4  #@param {type:"slider", min:0.5, max:4, step:0.1}
AXES_LINEWIDTH = 1.0  #@param {type:"slider", min:0.4, max:3, step:0.1}
FIG_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.05}
SHOW_REPLICATE_TRACES = True  #@param {type:"boolean"}
USE_LINESTYLES = True  #@param {type:"boolean"}
#@markdown *USE_LINESTYLES* añade un tipo de línea/trama distinto por sistema, además del color (recomendado: Ruby Leaf y Burnt Nectar son tonos cercanos).
HEATMAP_CMAP = "viridis"  #@param ["viridis", "cividis", "magma", "Blues", "YlGnBu"]
#@markdown **Análisis**
EQ_START_NS = 0.0  #@param {type:"number"}
SMOOTH_NS = 0.5  #@param {type:"number"}
NAC_DIST = 3.5  #@param {type:"number"}
NAC_ANGLE = 130.0  #@param {type:"number"}

### 🎨 Paleta y etiquetas de ejes
Edita los colores (HEX) y cualquier texto de ejes en los diccionarios de abajo.

> ⚠️ **Nota sobre la paleta:** en la imagen de referencia, el HEX y el RGB de cada color no coinciden. Usé el valor que corresponde
> al color que se ve en la muestra; la alternativa queda comentada por si prefieres la otra.

In [ ]:
PALETTE = {
    "WT":         "#0D244D",   # Indigo Rain  (HEX de la imagen; su RGB 13,62,77 sería #0D3E4D)
    "R53A":       "#852E47",   # Ruby Leaf    (HEX de la imagen; su RGB 182,46,71 sería #B62E47)
    "R272L":      "#4A5A30",   # Moss Shadow  (RGB 74,90,48; el HEX #AA542B de la imagen es naranja, claramente un error)
    "R53A/R272L": "#C2441C",   # Burnt Nectar (HEX de la imagen; su RGB 134,74,28 sería #864A1C)
}
LINESTYLES = {"WT": "-", "R53A": "--", "R272L": "-.", "R53A/R272L": ":"}
HATCHES    = {"WT": "", "R53A": "//", "R272L": "..", "R53A/R272L": "xx"}
EXTRA = {   # colores de figuras de un solo sistema (WT): categorías que no son mutantes
    "helix": "#0D244D", "sheet": "#8C8C8C",          # Fig. 3C
    "fad": "#8C8C8C", "glyphosate": "#0D244D",        # Fig. 4A
    "c_n5": "#0D244D", "h_n5": "#8C8C8C",              # Fig. 4B
}
REP_STYLES = ["-", "--", ":"]                          # réplicas 1–3 en Fig. S5 (mismo color del sistema)

AXIS_LABELS = {
    "time": "Time (ns)",
    "rmsd_bb": "Backbone RMSD (nm)",
    "rg": "Radius of gyration (nm)",
    "ss": "Secondary structure (%)",
    "rmsd_lig_fad": "RMSD (Å)",
    "dist": "Distance (Å)",
    "angle": "C1–H···N5 angle (°)",
    "c_n5": "C1···N5 distance (Å)",
    "occ_heat": "Interaction occupancy (%)",
    "dg": r"$\Delta G_{\mathrm{bind}}$, MM-GBSA (kcal mol$^{-1}$)",
    "dg_res": r"Per-residue $\Delta G$ (kcal mol$^{-1}$)",
    "r53_tc": "R53–phosphonate salt bridge\n(% of frames, 5-ns window)",
    "lig_rmsd": "Glyphosate RMSD (Å)",
    "helix": "α-Helix (%)",
    "contact": "Contact occupancy (%)",
}
LEGEND_LABELS = {"helix": "α-helix", "sheet": "β-sheet", "fad": "FAD", "glyphosate": "Glyphosate",
                 "c_n5": "C1···N5", "h_n5": "H···N5 (closer H)"}
print("Paleta:", PALETTE)

## 1 · Entorno, Google Drive y datos

In [ ]:
import os, glob, zipfile, shutil, subprocess, numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
try:
    from IPython.display import display
except ImportError:
    display = print

# --- Google Drive ---
OUT = os.path.abspath("go19_md_results_redo_local")
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
    except ImportError:
        print("No estoy en Colab: guardo localmente en", OUT)
FIGDIR, TABDIR, CAPDIR = (os.path.join(OUT, d) for d in ("figures", "tables", "captions"))
for d in (FIGDIR, TABDIR, CAPDIR):
    os.makedirs(d, exist_ok=True)
print("Carpeta de salida:", OUT)

# --- Datos: GO19_colab.zip en /content, en la carpeta de Drive, o subirlo ---
BASE = "GO19_results"
if not os.path.isdir(os.path.join(BASE, "results")):
    zp = next((p for p in ["GO19_colab.zip", os.path.join(OUT, "GO19_colab.zip")] if os.path.isfile(p)), None)
    if zp is None:
        from google.colab import files
        print("Selecciona GO19_colab.zip ...")
        zp = list(files.upload().keys())[0]
    with zipfile.ZipFile(zp) as z:
        z.extractall(BASE)
    if os.path.abspath(zp) != os.path.abspath(os.path.join(OUT, "GO19_colab.zip")):
        shutil.copy(zp, os.path.join(OUT, "GO19_colab.zip"))   # copia de respaldo en Drive
    print("Datos descomprimidos desde", zp)

SYS = {"go19_v2": "WT", "go19_v2_R53A": "R53A", "go19_v2_R272L": "R272L", "go19_v2_R53A_R272L": "R53A/R272L"}
LABELS = list(SYS.values())

## 2 · Estilo, formatos de salida y funciones auxiliares

In [ ]:
# --- Fuente: Arial/Helvetica/Times no vienen en Colab; se instalan sus equivalentes métricos (Liberation) ---
FONT_MAP = {"Arial": "Liberation Sans", "Helvetica": "Liberation Sans", "Times New Roman": "Liberation Serif"}
installed = {f.name for f in fm.fontManager.ttflist}
fam = FONT_FAMILY
if fam not in installed and fam in FONT_MAP:
    if FONT_MAP[fam] not in installed:
        subprocess.run("apt-get -qq install -y fonts-liberation > /dev/null 2>&1", shell=True)
        for f in glob.glob("/usr/share/fonts/truetype/liberation*/*.ttf"):
            fm.fontManager.addfont(f)
    if FONT_MAP[fam] in {f.name for f in fm.fontManager.ttflist}:
        print(f"{fam} no está disponible en Colab → uso {FONT_MAP[fam]} (mismas métricas).")
        fam = FONT_MAP[fam]
    else:
        print(f"No se pudo instalar un equivalente de {fam}; uso DejaVu Sans."); fam = "DejaVu Sans"

matplotlib.rcdefaults()
plt.rcParams.update({
    "figure.dpi": 100, "font.family": "sans-serif" if "Serif" not in fam else "serif",
    "font.sans-serif": [fam, "DejaVu Sans"], "font.serif": [fam, "DejaVu Serif"],
    "font.size": TICK_LABEL_SIZE, "axes.labelsize": AXIS_LABEL_SIZE, "axes.labelweight": AXIS_LABEL_WEIGHT,
    "xtick.labelsize": TICK_LABEL_SIZE, "ytick.labelsize": TICK_LABEL_SIZE,
    "legend.fontsize": LEGEND_SIZE, "legend.frameon": False, "axes.linewidth": AXES_LINEWIDTH,
    "xtick.major.width": AXES_LINEWIDTH, "ytick.major.width": AXES_LINEWIDTH,
    "savefig.bbox": "tight", "svg.fonttype": "none", "pdf.fonttype": 42, "ps.fonttype": 42,   # texto editable en SVG/PDF
})

FMT_SETS = {"png": ["png"], "tiff": ["tiff"], "svg": ["svg"], "pdf": ["pdf"], "png + tiff": ["png", "tiff"],
            "png + svg": ["png", "svg"], "png + pdf": ["png", "pdf"], "svg + pdf": ["svg", "pdf"],
            "all": ["png", "tiff", "svg", "pdf"]}
FORMATS = FMT_SETS[FORMAT]
DPIS = [int(x) for x in DPI_LIST.replace(";", ",").split(",") if x.strip()]
print("Formatos:", FORMATS, "| dpi (raster):", DPIS, "| fuente:", fam)

def save_fig(fig, name):
    saved = []
    for fmt in FORMATS:
        if fmt in ("svg", "pdf"):                                   # vectoriales: un solo archivo
            p = os.path.join(FIGDIR, f"{name}.{fmt}"); fig.savefig(p, format=fmt, pad_inches=0.15); saved.append(p)
        else:
            for dpi in DPIS:
                p = os.path.join(FIGDIR, f"{name}_{dpi}dpi.{fmt}")
                kw = {"pil_kwargs": {"compression": TIFF_COMPRESSION}} if fmt == "tiff" and TIFF_COMPRESSION != "none" else {}
                fig.savefig(p, format=fmt, dpi=dpi, pad_inches=0.15, **kw); saved.append(p)
    print("Guardado:", ", ".join(os.path.basename(p) for p in saved))

CAPTIONS = {}
def save_caption(name, text):
    CAPTIONS[name] = text.strip()
    with open(os.path.join(CAPDIR, f"{name}.txt"), "w", encoding="utf-8") as f:
        f.write(CAPTIONS[name] + "\n")
    with open(os.path.join(CAPDIR, "all_captions.md"), "w", encoding="utf-8") as f:
        for k, v in CAPTIONS.items():
            f.write(f"## {k}\n\n{v}\n\n")
    print("\n" + CAPTIONS[name])

def figsize(w, h):
    return (w * FIG_SCALE, h * FIG_SCALE)

def ls(label):
    return LINESTYLES.get(label, "-") if USE_LINESTYLES else "-"

def style_axes(ax, xkey=None, ykey=None):
    if xkey: ax.set_xlabel(AXIS_LABELS.get(xkey, xkey), fontsize=AXIS_LABEL_SIZE, fontweight=AXIS_LABEL_WEIGHT, fontstyle=AXIS_LABEL_STYLE)
    if ykey: ax.set_ylabel(AXIS_LABELS.get(ykey, ykey), fontsize=AXIS_LABEL_SIZE, fontweight=AXIS_LABEL_WEIGHT, fontstyle=AXIS_LABEL_STYLE)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight(TICK_LABEL_WEIGHT); t.set_fontsize(TICK_LABEL_SIZE)

def legend(ax, **kw):
    lg = ax.legend(prop={"size": LEGEND_SIZE, "weight": LEGEND_WEIGHT, "style": LEGEND_STYLE}, **kw)
    return lg

def tag(ax, letter):
    if PANEL_LETTERS:
        ax.text(-0.13, 1.04, letter, transform=ax.transAxes, fontsize=PANEL_LETTER_SIZE,
                fontweight=PANEL_LETTER_WEIGHT, fontstyle=PANEL_LETTER_STYLE)

def reps(s):
    return sorted(glob.glob(os.path.join(BASE, "results", s, "prod_rep*")))

def series(s, fname, col, k=1.0, smooth_ns=None):
    sm = SMOOTH_NS if smooth_ns is None else smooth_ns
    out = []
    for r in reps(s):
        d = pd.read_csv(os.path.join(r, fname)); t = d["time_ns"].to_numpy(float)
        y = d[col].to_numpy(float) * k
        win = int(round(sm / max(np.median(np.diff(t)), 1e-9)))
        if win > 1:
            y = pd.Series(y).rolling(win, center=True, min_periods=1).mean().to_numpy()
        out.append((t, y))
    return out

def band(ax, ser, color, label, linestyle="-", traces=None):
    traces = SHOW_REPLICATE_TRACES if traces is None else traces
    n = min(len(t) for t, _ in ser); t = ser[0][0][:n]; Y = np.vstack([y[:n] for _, y in ser])
    if traces:
        for y in Y:
            ax.plot(t, y, color=color, lw=0.4, alpha=0.35, ls="-")
    ax.fill_between(t, Y.min(0), Y.max(0), color=color, alpha=0.18, lw=0)
    ax.plot(t, Y.mean(0), color=color, lw=LINE_WIDTH, ls=linestyle, label=label)

def eq_means(s, fname, col, k=1.0):
    v = []
    for r in reps(s):
        d = pd.read_csv(os.path.join(r, fname)); d = d[d["time_ns"] >= EQ_START_NS]
        v.append(d[col].to_numpy(float).mean() * k)
    return np.array(v)

def msd(v, nd=2):
    v = np.asarray(v, float); return f"{v.mean():.{nd}f} ± {v.std(ddof=1):.{nd}f}"

## 3 · Control de calidad

In [ ]:
EXPECTED = ["rmsd_backbone.csv", "rmsd_fad_lig.csv", "rg.csv", "dssp_timeseries.csv", "contactos_lig.csv",
            "hydride_transfer_geometry.csv", "pca_variance.csv", "interacciones_prolif_raw.csv"]
rows = []
for s, l in SYS.items():
    for r in reps(s):
        t = pd.read_csv(os.path.join(r, "rmsd_backbone.csv"))["time_ns"].to_numpy()
        miss = [f for f in EXPECTED if not os.path.isfile(os.path.join(r, f))]
        rows.append({"system": l, "replica": os.path.basename(r), "n_frames": len(t), "t_end_ns": round(t[-1], 2),
                     "dt_ns": round(float(np.median(np.diff(t))), 4), "missing": ", ".join(miss) or "—"})
qc = pd.DataFrame(rows); display(qc)
for f in ["prolif_table1A_full.csv", "mmgbsa_per_rep.csv", "mmgbsa_decomp_per_rep.csv"]:
    print(("✅ " if os.path.isfile(os.path.join(BASE, f)) else "❌ falta ") + f)
assert len(qc) == 12, "Se esperaban 12 réplicas (4 sistemas × 3)"
assert (qc["missing"] == "—").all(), "Faltan archivos — revisa la tabla"
assert (qc["t_end_ns"] >= 99.5).all(), "Alguna réplica no llega a 100 ns"
print(f"✅ 12 réplicas completas · ventana analizada {EQ_START_NS:g}–100 ns")
for f in ["nac/nac_sensitivity_per_rep.csv", "equil_check/window_compare.csv"]:
    print(("✅ " if os.path.isfile(os.path.join(BASE, f)) else "⚠️ falta (tabla opcional) ") + f)

## 4 · Tabla resumen (estabilidad y geometría catalítica)

In [ ]:
def nac_pct(s):
    v = []
    for r in reps(s):
        d = pd.read_csv(os.path.join(r, "hydride_transfer_geometry.csv")); d = d[d["time_ns"] >= EQ_START_NS]
        v.append(100 * ((d["dist_C_N5_A"] <= NAC_DIST) & (d["angle_C_H_N5_deg"] >= NAC_ANGLE)).mean())
    return np.array(v)
rows = []
for s, l in SYS.items():
    rows.append({"System": l,
        "Backbone RMSD (nm)": msd(eq_means(s, "rmsd_backbone.csv", "rmsd_backbone_A", 0.1)),
        "FAD RMSD (Å)": msd(eq_means(s, "rmsd_fad_lig.csv", "rmsd_fad_A")),
        "Glyphosate RMSD (Å)": msd(eq_means(s, "rmsd_fad_lig.csv", "rmsd_lig_A")),
        "Rg (nm)": msd(eq_means(s, "rg.csv", "rg_nm")),
        "α-helix (%)": msd(eq_means(s, "dssp_timeseries.csv", "pct_Helice_alfa"), 1),
        "β-sheet (%)": msd(eq_means(s, "dssp_timeseries.csv", "pct_Hoja_beta"), 1),
        "C1–N5 (Å)": msd(eq_means(s, "hydride_transfer_geometry.csv", "dist_C_N5_A")),
        "H···N5 (Å)": msd(eq_means(s, "hydride_transfer_geometry.csv", "dist_closerH_N5_A")),
        "C–H···N5 (°)": msd(eq_means(s, "hydride_transfer_geometry.csv", "angle_C_H_N5_deg"), 1),
        "NAC (%)": msd(nac_pct(s), 1)})
summary = pd.DataFrame(rows).set_index("System"); display(summary.T)
summary.to_csv(os.path.join(TABDIR, "TableS_MD_summary.csv"))
print(f"NAC = frames con C1–N5 ≤ {NAC_DIST} Å y C–H···N5 ≥ {NAC_ANGLE}°; media ± SD (n−1) de 3 réplicas, {EQ_START_NS:g}–100 ns.")
print("Valores esperados (PC, 0–100 ns) — WT: C1–N5 3.48 ± 0.07 Å, ángulo 112.0 ± 4.0°, NAC 7.5 ± 5.6 %")

## 5 · Figura 3 — Estabilidad estructural (WT)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=figsize(13, 3.6))
c = PALETTE["WT"]
band(ax[0], series("go19_v2", "rmsd_backbone.csv", "rmsd_backbone_A", 0.1), c, "WT"); style_axes(ax[0], "time", "rmsd_bb")
band(ax[1], series("go19_v2", "rg.csv", "rg_nm"), c, "WT"); style_axes(ax[1], "time", "rg")
band(ax[2], series("go19_v2", "dssp_timeseries.csv", "pct_Helice_alfa"), EXTRA["helix"], LEGEND_LABELS["helix"], "-")
band(ax[2], series("go19_v2", "dssp_timeseries.csv", "pct_Hoja_beta"), EXTRA["sheet"], LEGEND_LABELS["sheet"], "--")
style_axes(ax[2], "time", "ss"); legend(ax[2], loc="center right")
for a, l in zip(ax, "ABC"):
    a.set_xlim(0, 100); tag(a, l)
fig.tight_layout(); save_fig(fig, "Fig3_WT_stability"); plt.show()
save_caption("Fig3_WT_stability", f"""
Figure 3. Structural stability of the GO19–glyphosate–FAD complex across three independent 100-ns MD replicates.
(A) Protein backbone RMSD relative to the starting structure. (B) Radius of gyration of the protein. (C) α-Helix and β-sheet
content assigned with DSSP. In each panel the bold line is the mean of the three replicates, the shaded band is the range
(minimum–maximum) between replicates and the thin lines are the individual replicates; curves were smoothed with a
{SMOOTH_NS:g}-ns running mean. The fold remains compact and its secondary structure unchanged throughout all replicates.""")

## 6 · Figura 4 — Sitio activo y geometría de transferencia de hidruro (WT)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=figsize(13, 3.6))
band(ax[0], series("go19_v2", "rmsd_fad_lig.csv", "rmsd_fad_A"), EXTRA["fad"], LEGEND_LABELS["fad"], "--")
band(ax[0], series("go19_v2", "rmsd_fad_lig.csv", "rmsd_lig_A"), EXTRA["glyphosate"], LEGEND_LABELS["glyphosate"], "-")
style_axes(ax[0], "time", "rmsd_lig_fad"); legend(ax[0], loc="best")
band(ax[1], series("go19_v2", "hydride_transfer_geometry.csv", "dist_C_N5_A"), EXTRA["c_n5"], LEGEND_LABELS["c_n5"], "-")
band(ax[1], series("go19_v2", "hydride_transfer_geometry.csv", "dist_closerH_N5_A"), EXTRA["h_n5"], LEGEND_LABELS["h_n5"], "--")
ax[1].axhline(NAC_DIST, color="grey", ls="--", lw=0.8); style_axes(ax[1], "time", "dist"); legend(ax[1], loc="best")
band(ax[2], series("go19_v2", "hydride_transfer_geometry.csv", "angle_C_H_N5_deg"), PALETTE["WT"], "WT")
ax[2].axhline(NAC_ANGLE, color="grey", ls="--", lw=0.8); ax[2].axhline(180, color="grey", ls=":", lw=0.8)
style_axes(ax[2], "time", "angle")
for a, l in zip(ax, "ABC"):
    a.set_xlim(0, 100); tag(a, l)
fig.tight_layout(); save_fig(fig, "Fig4_WT_active_site"); plt.show()
save_caption("Fig4_WT_active_site", f"""
Figure 4. Active-site dynamics of the GO19–glyphosate–FAD complex. (A) RMSD of FAD and glyphosate after superposition on the
protein backbone. (B) Distance from the glyphosate reactive carbon C1 (the methylene oxidised to glyoxylate) and from its closer
hydrogen to the flavin N5 acceptor; dashed line, {NAC_DIST:g} Å. (C) C1–H···N5 angle; dashed line, {NAC_ANGLE:g}°; dotted line,
linear geometry (180°). Glyphosate and FAD remain bound in all replicates and C1 stays close to N5, but the C–H···N5 alignment is
mostly far from linear: near-attack conformations (C1–N5 ≤ {NAC_DIST:g} Å and C–H···N5 ≥ {NAC_ANGLE:g}°) are sampled in a minority of
frames. Lines, band and smoothing as in Figure 3.""")

## 7 · Figura 9 — WT frente a mutantes

In [ ]:
pl = pd.read_csv(os.path.join(BASE, "prolif_table1A_full.csv"))
g = pd.read_csv(os.path.join(BASE, "mmgbsa_per_rep.csv"))
dc = pd.read_csv(os.path.join(BASE, "mmgbsa_decomp_per_rep.csv"))

fig, ax = plt.subplots(2, 2, figsize=figsize(12, 8.5)); ax = ax.ravel()
for s, l in SYS.items():
    band(ax[0], series(s, "hydride_transfer_geometry.csv", "dist_C_N5_A", smooth_ns=2.0), PALETTE[l], l, ls(l), traces=False)
ax[0].axhline(NAC_DIST, color="grey", ls="--", lw=0.8); ax[0].set_xlim(0, 100)
style_axes(ax[0], "time", "c_n5"); legend(ax[0], ncol=2, loc="upper left")

HM = [53, 257, 272, 314, 341, 251, 54]
HM_NAME = {53: "R53", 257: "Y257", 272: "R272", 314: "R314", 341: "P341", 251: "S251", 54: "D54"}
anyi = pl[pl["interaction"] == "ANY"]
H = np.array([[anyi[(anyi["system"] == l) & (anyi["num"] == n)]["mean"].sum() for l in LABELS] for n in HM])
im = ax[1].imshow(H, cmap=HEATMAP_CMAP, vmin=0, vmax=100, aspect="auto")
cm = plt.get_cmap(HEATMAP_CMAP)
for i in range(len(HM)):
    for j in range(len(LABELS)):
        r, gg, b, _ = cm(H[i, j] / 100); lum = 0.299 * r + 0.587 * gg + 0.114 * b
        ax[1].text(j, i, f"{H[i, j]:.0f}", ha="center", va="center", fontsize=TICK_LABEL_SIZE, fontweight=TICK_LABEL_WEIGHT,
                   color="white" if lum < 0.5 else "black")
ax[1].set_xticks(range(len(LABELS))); ax[1].set_xticklabels(LABELS)
ax[1].set_yticks(range(len(HM))); ax[1].set_yticklabels([HM_NAME[n] for n in HM]); style_axes(ax[1])
cb = fig.colorbar(im, ax=ax[1], fraction=0.04)
cb.set_label(AXIS_LABELS["occ_heat"], fontsize=AXIS_LABEL_SIZE, fontweight=AXIS_LABEL_WEIGHT, fontstyle=AXIS_LABEL_STYLE)
cb.ax.tick_params(labelsize=TICK_LABEL_SIZE)

for j, l in enumerate(LABELS):
    v = g[g["system"] == l]["DELTA TOTAL"].to_numpy()
    ax[2].bar(j, v.mean(), yerr=v.std(ddof=1), color=PALETTE[l], alpha=0.85, capsize=4, width=0.6,
              hatch=HATCHES[l] if USE_LINESTYLES else "", edgecolor="white", linewidth=0)
    ax[2].scatter(j + np.linspace(-0.12, 0.12, len(v)), v, color="white", edgecolor="black", zorder=3, s=22)
ax[2].set_xticks(range(len(LABELS))); ax[2].set_xticklabels(LABELS); ax[2].axhline(0, color="black", lw=0.8)
style_axes(ax[2], None, "dg")

DR = [53, 257, 272, 314, 341, 385, 54]
DR_NAME = {53: "R53", 257: "Y257", 272: "R272", 314: "R314", 341: "P341", 385: "FAD", 54: "D54"}
w = 0.2
for j, l in enumerate(LABELS):
    st = dc[dc["system"] == l].groupby("num")["total"].agg(["mean", lambda x: x.std(ddof=1)]).reindex(DR).fillna(0)
    ax[3].bar(np.arange(len(DR)) + (j - 1.5) * w, st["mean"], yerr=st.iloc[:, 1], width=w, color=PALETTE[l], label=l,
              capsize=2, hatch=HATCHES[l] if USE_LINESTYLES else "", edgecolor="white", linewidth=0)
ax[3].set_xticks(range(len(DR))); ax[3].set_xticklabels([DR_NAME[r] for r in DR]); ax[3].axhline(0, color="black", lw=0.8)
style_axes(ax[3], None, "dg_res"); legend(ax[3], ncol=2, loc="lower right")
for a, l in zip(ax, "ABCD"):
    tag(a, l)
fig.tight_layout(); save_fig(fig, "Fig9_WT_vs_mutants"); plt.show()
save_caption("Fig9_WT_vs_mutants", f"""
Figure 9. Effect of the R53A, R272L and R53A/R272L substitutions on glyphosate binding. (A) C1···N5 distance in the wild type
and the three mutants (mean of three 100-ns replicates; band, range between replicates; 2-ns running mean); dashed line,
{NAC_DIST:g} Å. Glyphosate remains bound with C1 close to N5 in every system. (B) Occupancy (% of frames over the full 100-ns trajectories, mean
of three replicates) of any ProLIF-detected interaction between glyphosate and active-site residues; the R53 and R272 rows refer to
Ala53 and Leu272 in the corresponding mutants. (C) MM-GBSA binding free energy (bars, mean ± SD of replicate means; circles,
individual replicates). (D) Per-residue MM-GBSA decomposition for the main contributors (mean ± SD). Removing R53 weakens
binding, removing R272 alone has no detectable effect, and removing both weakens binding most, while R314 and Y257 contribute
equally in all systems. Absolute MM-GBSA values for a dianionic ligand are not quantitative and are interpreted only
comparatively.""")

## 8 · Tablas 1A (ProLIF) y 1B (MM-GBSA)

In [ ]:
KEY = [(53, "R53"), (257, "Y257"), (272, "R272"), (314, "R314"), (341, "P341"), (251, "S251")]
INTER = {"ANY": "Any", "Anionic": "Ionic (salt bridge)", "HBAcceptor": "H-bond (glyphosate acceptor)",
         "HBDonor": "H-bond (glyphosate donor)", "VdWContact": "van der Waals"}
rows = []
for n, name in KEY:
    for i, iname in INTER.items():
        sub = pl[(pl["num"] == n) & (pl["interaction"] == i)]
        if len(sub) == 0 or sub["mean"].max() < 1:
            continue
        row = {"Residue": name, "Interaction": iname}
        for l in LABELS:
            x = sub[sub["system"] == l]
            row[l] = "0" if len(x) == 0 else f"{x['mean'].iloc[0]:.0f} ± {x['sd'].iloc[0]:.0f} ({x['min'].iloc[0]:.0f}–{x['max'].iloc[0]:.0f})"
        rows.append(row)
t1a = pd.DataFrame(rows); display(t1a); t1a.to_csv(os.path.join(TABDIR, "Table1A_ProLIF.csv"), index=False)

TERMS = {"VDWAALS": "ΔE_vdW", "EEL": "ΔE_elec", "EGB": "ΔG_GB", "ESURF": "ΔG_SA", "DELTA TOTAL": "ΔG_bind"}
wt = g[g["system"] == "WT"]["DELTA TOTAL"].to_numpy(); rows = []
for l in LABELS:
    x = g[g["system"] == l]; row = {"System": l}
    for k, v in TERMS.items():
        row[v] = msd(x[k])
    row["Replicates (ΔG_bind)"] = ", ".join(f"{v:.1f}" for v in x["DELTA TOTAL"])
    y = x["DELTA TOTAL"].to_numpy()
    row["ΔΔG vs WT (± SE)"] = "—" if l == "WT" else f"{y.mean()-wt.mean():+.1f} ± {np.sqrt(y.var(ddof=1)/len(y) + wt.var(ddof=1)/len(wt)):.1f}"
    rows.append(row)
t1b = pd.DataFrame(rows); display(t1b); t1b.to_csv(os.path.join(TABDIR, "Table1B_MMGBSA.csv"), index=False)

st = dc.groupby(["system", "num"])["total"].agg(["mean", lambda x: x.std(ddof=1)]).reset_index(); st.columns = ["system", "num", "mean", "sd"]
names = dc[dc["system"] == "WT"].drop_duplicates("num").set_index("num")["resname"].to_dict()
keep = st.groupby("num")["mean"].apply(lambda v: v.abs().max() >= 0.5); keep = keep[keep].index
dec = st[st["num"].isin(keep)].assign(v=lambda x: x["mean"].map("{:+.2f}".format) + " ± " + x["sd"].map("{:.2f}".format)) \
        .pivot(index="num", columns="system", values="v").reindex(columns=LABELS)
dec.index = [f"{names.get(n, '?')}{n}" for n in dec.index]
display(dec); dec.to_csv(os.path.join(TABDIR, "TableS_MMGBSA_decomposition.csv"))
save_caption("Table1_notes", f"""
Table 1. (A) Occupancy (% of frames over the full 100-ns trajectories, every 100 ps; 1001 frames per replicate) of glyphosate–protein interactions detected with ProLIF,
mean ± SD (minimum–maximum) of three independent 100-ns replicates per system. Hydrogen bonds: donor–acceptor ≤ 3.5 Å and
D–H···A ≥ 130°; ionic interactions: cation–anion ≤ 4.5 Å. (B) MM-GBSA binding free energy (kcal mol⁻¹; single-trajectory,
GB model igb = 5, mbondi2 radii, 0.15 M salt, 251 frames per replicate taken every 400 ps over the full 100-ns trajectories), mean ± SD of replicate means;
ΔΔG relative to the wild type with combined standard error. Entropy was not computed; absolute values for a dianionic ligand are
interpreted only comparatively.""")

## 9 · Figura S5 — Curso temporal del puente salino R53–fosfonato

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=figsize(11, 3.4), sharey=True)
for a, (s, l) in zip(ax, [("go19_v2", "WT"), ("go19_v2_R272L", "R272L")]):
    for k, r in enumerate(reps(s)):
        df = pd.read_csv(os.path.join(r, "interacciones_prolif_raw.csv"), header=[0, 1, 2], index_col=0).astype(bool)
        cols = [c for c in df.columns if str(c[1]).startswith("ARG53") and c[2] == "Anionic"]
        occ = df[cols].any(axis=1).astype(float).rolling(50, center=True, min_periods=1).mean() * 100   # 50 × 100 ps = 5 ns
        t = pd.to_numeric(pd.Series(df.index), errors="coerce").to_numpy() * 0.01                    # índice de frame × 10 ps
        a.plot(t, occ.to_numpy(), lw=LINE_WIDTH * 0.8, color=PALETTE[l], ls=REP_STYLES[k], label=f"Replicate {k + 1}")
    a.set_title(l, fontsize=AXIS_LABEL_SIZE, fontweight=AXIS_LABEL_WEIGHT, fontstyle=AXIS_LABEL_STYLE)
    a.set_xlim(0, 100); a.set_ylim(-5, 105); style_axes(a, "time", None); legend(a, loc="lower left")
style_axes(ax[0], None, "r53_tc")
for a, l in zip(ax, "AB"):
    tag(a, l)
fig.tight_layout(); save_fig(fig, "FigS5_R53_timecourse"); plt.show()
save_caption("FigS5_R53_timecourse", """
Figure S5. Time course of the R53–phosphonate salt bridge. Percentage of frames with an ionic interaction between the R53
guanidinium and the glyphosate phosphonate (ProLIF; 5-ns running window) in each replicate of (A) the wild type and (B) R272L.
The interaction is present from the start of production and persists in five of six replicates; in wild-type replicate 1 it
breaks at ~60–80 ns and does not re-form within the simulated time, showing that the contact is dominant but reversible.""")

## 10 · Figura S6 — Estabilidad de los mutantes

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=figsize(11, 7)); ax = ax.ravel()
spec = [("rmsd_backbone.csv", "rmsd_backbone_A", 0.1, "rmsd_bb"), ("rg.csv", "rg_nm", 1.0, "rg"),
        ("rmsd_fad_lig.csv", "rmsd_lig_A", 1.0, "lig_rmsd"), ("dssp_timeseries.csv", "pct_Helice_alfa", 1.0, "helix")]
for a, (f, c, k, yk), letter in zip(ax, spec, "ABCD"):
    for s, l in SYS.items():
        band(a, series(s, f, c, k, smooth_ns=1.0), PALETTE[l], l, ls(l), traces=False)
    a.set_xlim(0, 100); style_axes(a, "time", yk); tag(a, letter)
legend(ax[0], ncol=2)
fig.tight_layout(); save_fig(fig, "FigS6_mutant_stability"); plt.show()
save_caption("FigS6_mutant_stability", """
Figure S6. Global stability of the wild type and the R53A, R272L and R53A/R272L mutants. (A) Backbone RMSD, (B) radius of
gyration, (C) glyphosate RMSD after superposition on the protein backbone and (D) α-helix content (DSSP). Lines, mean of three
100-ns replicates; bands, range between replicates; 1-ns running mean. None of the substitutions perturbs the fold or releases
the ligand.""")

## 11 · Figura S7 — Ocupación de contactos proteína–glifosato

In [ ]:
occ, names = {}, {}
for s, l in SYS.items():
    per = []
    for r in reps(s):
        d = pd.read_csv(os.path.join(r, "contactos_lig.csv")); per.append(d.set_index("resid")["occupancy_pct"])
        if l == "WT":
            names.update(dict(zip(d["resid"], d["resname"])))
    occ[l] = pd.concat(per, axis=1).fillna(0)
mean = pd.DataFrame({l: o.mean(1) for l, o in occ.items()}).fillna(0)
sd = pd.DataFrame({l: o.std(1, ddof=1) for l, o in occ.items()}).fillna(0)
top = mean[(mean >= 30).any(axis=1)].index.sort_values()
fig, ax = plt.subplots(figsize=figsize(10, 3.8)); w = 0.2
for j, l in enumerate(LABELS):
    ax.bar(np.arange(len(top)) + (j - 1.5) * w, mean.loc[top, l], yerr=sd.loc[top, l], width=w, color=PALETTE[l], label=l,
           capsize=2, hatch=HATCHES[l] if USE_LINESTYLES else "", edgecolor="white", linewidth=0)
ax.set_xticks(range(len(top))); ax.set_xticklabels([f"{names.get(n, '?')}{n}" for n in top])
ax.set_ylim(0, 110); style_axes(ax, None, "contact"); legend(ax, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.15))
fig.tight_layout(); save_fig(fig, "FigS7_contact_occupancy"); plt.show()
save_caption("FigS7_contact_occupancy", """
Figure S7. Protein–glyphosate contact occupancy. Percentage of frames in which any heavy atom of the residue lies within 4.5 Å
of a glyphosate heavy atom, for residues reaching ≥30% in at least one system (mean ± SD of three replicates). Residue labels
refer to the wild type; R53 and R272 correspond to Ala53 and Leu272 in the mutants.""")

## 11b · Tabla S5 — Sensibilidad de las conformaciones de casi-ataque (NAC) a los umbrales

In [ ]:
p = os.path.join(BASE, "nac", "nac_sensitivity_per_rep.csv")
if not os.path.isfile(p):
    print("⚠️ falta nac/nac_sensitivity_per_rep.csv en el zip — se omite la Tabla S5")
else:
    nac = pd.read_csv(p)
    assert (nac["frames"] == 10001).all(), "La tabla NAC no es 0–100 ns (se esperaban 10001 frames por réplica)"
    GRID = [(3.3, 140), (3.5, 130), (3.7, 120), (4.0, 120), (3.5, 150), (4.0, 150)]
    rows = []
    for D, A in GRID:
        row = {"C1–N5 ≤ (Å)": D, "C–H···N5 ≥ (°)": A}
        for s, l in SYS.items():
            row[l] = msd(nac[nac["system"] == s][f"close_{D}_{A}"], 1)
        rows.append(row)
    ts5 = pd.DataFrame(rows); display(ts5); ts5.to_csv(os.path.join(TABDIR, "TableS5_NAC_sensitivity.csv"), index=False)
    save_caption("TableS5_NAC_sensitivity", """
Table S5. Sensitivity of the near-attack-conformation (NAC) frequency to the geometric criteria. Percentage of frames in which
the glyphosate reactive carbon C1 lies within the stated distance of FAD N5 and the C1–H···N5 angle (closer hydrogen) exceeds the
stated value, over the full 100-ns trajectories (10,001 frames per replicate); mean ± SD of three replicates. The reference
criteria used in the main text are 3.5 Å and 130°.""")

## 11c · Tabla S6 — Efecto de excluir los primeros 20 ns

In [ ]:
p = os.path.join(BASE, "equil_check", "window_compare.csv")
if not os.path.isfile(p):
    print("⚠️ falta equil_check/window_compare.csv en el zip — se omite la Tabla S6")
else:
    wc = pd.read_csv(p).replace(SYS)
    wc = wc.rename(columns={"system": "System", "observable": "Observable", "mean_20_100": "20–100 ns",
                            "mean_0_100": "0–100 ns", "shift_0_100_vs_20_100": "Shift", "shift_in_SD_units": "Shift (SD units)"})
    wc = wc[["System", "Observable", "0–100 ns", "20–100 ns", "Shift", "Shift (SD units)"]]
    display(wc); wc.to_csv(os.path.join(TABDIR, "TableS6_window_comparison.csv"), index=False)
    save_caption("TableS6_window_comparison", """
Table S6. Effect of excluding the first 20 ns of production on the reported averages. Means ± SD of three replicates over the full
trajectories (0–100 ns) and over 20–100 ns; "Shift" is the difference between the two, also expressed in units of the
between-replicate SD. Excluding the first 20 ns changed no average by more than 0.09 Å, 1.1° or 0.01 nm and did not alter the
ranking between systems.""")

## 12 · Resumen de lo guardado en Drive

In [ ]:
for d in (FIGDIR, TABDIR, CAPDIR):
    fs = sorted(os.listdir(d)); print(f"{d}  ({len(fs)} archivos)")
    for f in fs:
        print("   ", f)
print("\nPies de figura combinados:", os.path.join(CAPDIR, "all_captions.md"))

## Notas
- **Figura 5** (pose representativa) y **Figura S4** (MDpocket) se hacen en ChimeraX (`GO19_FigS4_mdpocket.cxc`). Para la Figura 5
  usa la réplica 2 o 3 del WT (o un centroide de clúster), **no** el último frame de la réplica 1.
- SVG y PDF guardan el texto como texto editable (Illustrator/Inkscape). TIFF con compresión LZW.
- Si cambias la paleta, revisa que los colores sigan distinguiéndose en escala de grises / daltonismo; por eso `USE_LINESTYLES`
  añade tipo de línea y trama por sistema.